In [1]:
# Import necessary libraries
import csv
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd

import scanpy as sc
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from model.dataset import BagsDataset, custom_collate_fn
from model.model import MIL, EarlyStopping


In [2]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    gpu_name = torch.cuda.get_device_name(torch.cuda.current_device())
    print(f"Using device: {device} ({gpu_name})")
else:
    print(f"Using device: {device}")
print("=====================================")


Using device: cpu


In [3]:

# Functions to load gene lists
def load_all_genes(reference_gene_file):
    
    all_genes = pd.read_csv(reference_gene_file)
    all_genes = all_genes['Gene'].values.tolist()
    
    return all_genes


In [4]:
# Set parameters (replace these with your own paths and settings)
# Paths to data and model
data_path = '/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/VisiumHD/HumanColorectalCancer'
reference_gene_path = 'data/human_filtered.csv'
pretrained_gene_path = 'data/human_filtered.csv'  # Pre-trained gene list
output_dir = os.path.join('test', data_path.split('/')[-1])
model_path = 'finalize_model/tcell/best_model.pth'  # Set to None if training from scratch


In [5]:

# Training parameters
immune_cell = 'tcell'       # Type of immune cell to consider
learning_rate = 1e-3      # Learning rate for the optimizer
num_epochs = 100          # Number of epochs to train the model
patience = 5                # Patience for early stopping
delta = 0.001               # Minimum change to qualify as an improvement
max_instances = None        # Maximum instances for the dataset
n_genes = 10000             # Number of genes to consider


In [6]:

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Load fine-tuning gene list
all_genes = load_all_genes(reference_gene_path)
print('Reference genes loaded:', len(all_genes))
print("=====================================")


Reference genes loaded: 23182


In [7]:

# Load pre-trained gene list
pretrained_genes = load_all_genes(pretrained_gene_path)
print('Pre-trained genes loaded:', len(pretrained_genes))


Pre-trained genes loaded: 23182


In [8]:
common_genes = list(set(pretrained_genes) & set(all_genes))
print(f'Number of common genes: {len(common_genes)}')

Number of common genes: 23182


In [9]:

# Create gene name to index mappings
pretrained_gene_to_index = {gene: idx for idx, gene in enumerate(pretrained_genes)}
fine_tuning_gene_to_index = {gene: idx for idx, gene in enumerate(all_genes)}


In [10]:

# Initialize the model
model = MIL(all_genes).to(device)


In [11]:

# Initialize a new tensor for immunogenicity.ig
new_ig_tensor = model.immunogenicity.ig.data.clone()


In [12]:

# Load pre-trained model's state dict
pretrained_state_dict = torch.load(model_path, map_location=device)


In [13]:

# Get the pre-trained immunogenicity.ig tensor
pretrained_ig_tensor = pretrained_state_dict['immunogenicity.ig']


In [14]:

# Copy over the values for common genes
for gene in common_genes:
    pretrained_idx = pretrained_gene_to_index[gene]
    fine_tuning_idx = fine_tuning_gene_to_index[gene]
    new_ig_tensor[fine_tuning_idx] = pretrained_ig_tensor[pretrained_idx]

# Assign the new tensor to the model
with torch.no_grad():
    model.immunogenicity.ig.copy_(new_ig_tensor)

print("Copied immunogenicity.ig values for common genes.")


Copied immunogenicity.ig values for common genes.


In [15]:

# Remove immunogenicity.ig from the pre-trained state dict
pretrained_state_dict.pop('immunogenicity.ig', None)


tensor([-1.0000, -0.9457, -1.0002,  ..., -4.6679,  0.9460, -1.0000])

In [16]:

# Load other compatible parameters
model_state_dict = model.state_dict()
common_keys = [k for k in pretrained_state_dict.keys()
               if k in model_state_dict and pretrained_state_dict[k].size() == model_state_dict[k].size()]
filtered_pretrained_state_dict = {k: pretrained_state_dict[k] for k in common_keys}
model_state_dict.update(filtered_pretrained_state_dict)
model.load_state_dict(model_state_dict)

print(f"Loaded matching model weights from {model_path} (excluding immunogenicity.ig).")


Loaded matching model weights from finalize_model/tcell/best_model.pth (excluding immunogenicity.ig).


In [17]:
model.state_dict()

OrderedDict([('alpha', tensor(1.7811)),
             ('beta', tensor(-2.0637)),
             ('distance.a', tensor(0.1668)),
             ('gene_expression.b', tensor(1.3164)),
             ('immunogenicity.ig',
              tensor([-1.0000, -0.9457, -1.0002,  ..., -4.6679,  0.9460, -1.0000]))])

In [18]:
"""model.alpha = nn.Parameter(torch.tensor(1.0), requires_grad=True)
model.beta = nn.Parameter(torch.tensor(1.0), requires_grad=True)
model.distance.a = nn.Parameter(torch.tensor(1.0), requires_grad=True)
model.gene_expression.b = nn.Parameter(torch.tensor(1.0), requires_grad=True)
"""

'model.alpha = nn.Parameter(torch.tensor(1.0), requires_grad=True)\nmodel.beta = nn.Parameter(torch.tensor(1.0), requires_grad=True)\nmodel.distance.a = nn.Parameter(torch.tensor(1.0), requires_grad=True)\nmodel.gene_expression.b = nn.Parameter(torch.tensor(1.0), requires_grad=True)\n'

In [19]:
model.state_dict()

OrderedDict([('alpha', tensor(1.7811)),
             ('beta', tensor(-2.0637)),
             ('distance.a', tensor(0.1668)),
             ('gene_expression.b', tensor(1.3164)),
             ('immunogenicity.ig',
              tensor([-1.0000, -0.9457, -1.0002,  ..., -4.6679,  0.9460, -1.0000]))])

In [21]:

# Optionally freeze pre-trained parameters (excluding immunogenicity.ig)
# for name, param in model.named_parameters():
#     if name in filtered_pretrained_state_dict:
#         param.requires_grad = False

# Define loss criterion and optimizer
criterion = nn.BCELoss().to(device)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)


In [22]:

# Load dataset
# Replace 'BagsDataset' and 'custom_collate_fn' with your data loading functions
adata = sc.read(os.path.join(data_path, 'T_cell.h5ad'))
sc.pp.filter_cells(adata, min_genes=1)
dataset = BagsDataset(adata, immune_cell=immune_cell, max_instances=max_instances, n_genes=18085, radius=75, resolution='high')
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)


['1', '0', '2']
Categories (3, object): ['0', '1', '2']
Tumor cells shape after filtering: (67054, 18085)
Selecting top 18085 genes based on mean expression
Top 18085 genes: Index(['MT-ND4', 'MT-ATP6', 'MT-CO3', 'TMSB4X', 'MT-CO2', 'MT-CYB', 'FTH1',
       'MT-ND3', 'CEACAM5', 'MUC12',
       ...
       'HNRNPCL3', 'PAGE2', 'BPIFB3', 'PRH2', 'LRRC70', 'APOBEC3B', 'CCDC197',
       'BPIFB6', 'ANKRD30B', 'C13orf46'],
      dtype='object', length=18085)
Preprocessed data: (126768, 18085)


Creating Bags with radius 75: 100%|███████████████████████| 126768/126768 [01:00<00:00, 2104.62it/s]

Total bags created: 70785
Average instances per bag: 51


In [23]:
train_labels_count = sum(1 for _, _, label, _, _, _ in train_loader if label.item() == 1)

# Count label=1 bags in validation dataset
val_labels_count = sum(1 for _, _, label, _, _, _ in val_loader if label.item() == 1)

# Print results
print(f"Number of bags with label 1 in train dataset: {train_labels_count}")
print(f"Number of bags with label 1 in validation dataset: {val_labels_count}")

Number of bags with label 1 in train dataset: 2765
Number of bags with label 1 in validation dataset: 1132


In [24]:
from torch.utils.data import Subset
import random

def balance_dataset(dataset, label_ratio=5):
    # Separate bags by labels
    label_0_indices = []
    label_1_indices = []

    for idx in range(len(dataset)):
        _, _, label, _, _, _ = dataset[idx]
        if label.item() == 0:
            label_0_indices.append(idx)
        else:
            label_1_indices.append(idx)

    # Calculate the number of label=0 bags to keep
    num_label_1 = len(label_1_indices)
    num_label_0_to_keep = min(len(label_0_indices), label_ratio * num_label_1)

    # Randomly sample label=0 indices
    sampled_label_0_indices = random.sample(label_0_indices, num_label_0_to_keep)

    # Combine sampled label=0 indices and all label=1 indices
    balanced_indices = sampled_label_0_indices + label_1_indices

    # Shuffle the combined indices
    random.shuffle(balanced_indices)

    # Return a Subset of the original dataset
    return Subset(dataset, balanced_indices)

# Balance train and validation datasets
balanced_train_dataset = balance_dataset(train_dataset, label_ratio=5)
balanced_val_dataset = balance_dataset(val_dataset, label_ratio=5)

# Create new DataLoaders with balanced datasets
train_loader = DataLoader(balanced_train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(balanced_val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)

# Print the new label distribution
train_labels_count = sum(1 for _, _, label, _, _, _ in train_loader if label.item() == 1)
train_label_0_count = sum(1 for _, _, label, _, _, _ in train_loader if label.item() == 0)
val_labels_count = sum(1 for _, _, label, _, _, _ in val_loader if label.item() == 1)
val_label_0_count = sum(1 for _, _, label, _, _, _ in val_loader if label.item() == 0)

print(f"Train dataset - Negative: {train_label_0_count}, Positive: {train_labels_count}")
print(f"Validation dataset - Negative: {val_label_0_count}, Postive: {val_labels_count}")


Train dataset - Negative: 13825, Positive: 2765
Validation dataset - Negative: 5660, Postive: 1132


In [27]:

# Initialize early stopping
early_stopping = EarlyStopping(patience=patience, delta=delta)

# Store IG scores before training
ig_scores_before_training = model.immunogenicity.ig.clone().detach().cpu()



In [28]:
ig_scores_before_training

tensor([-1.0000, -0.9457, -1.0002,  ..., -4.6679,  0.9460, -1.0000])

In [29]:
# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    gradients = {}

    with tqdm(train_loader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx, gene_names, cell_ids) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()

            distances = torch.stack(distances).to(device)
            gene_expressions = torch.stack(gene_expressions).to(device)
            label = label.clone().detach().float().to(device)

            output = model(distances, gene_expressions, list(gene_names[0]))

            loss = criterion(output, label)
            loss.backward()
            
            # Checking gradients
            for name, param in model.named_parameters():
                if param.grad is not None:
                    if name not in gradients:
                        gradients[name] = []
                    gradients[name].append(param.grad.cpu().numpy())

            optimizer.step()

            running_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(train_loader)
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

    # Validation
    model.eval()
    val_loss = 0.0
    val_predictions = []
    val_labels = []
    with torch.no_grad():
        for val_distances, val_gene_expressions, val_label, _, val_gene_names, val_cell_ids in val_loader:
            val_distances = torch.stack(val_distances).to(device)
            val_gene_expressions = torch.stack(val_gene_expressions).to(device)
            val_label = val_label.clone().detach().float().to(device)
            val_output = model(val_distances, val_gene_expressions, list(val_gene_names[0]))
            val_loss += criterion(val_output, val_label).item()
            val_predictions.extend(val_output.cpu().numpy())
            val_labels.extend(val_label.cpu().numpy())

    val_loss /= len(val_loader)
    val_auroc = roc_auc_score(val_labels, val_predictions)
    print(f'Validation Loss: {val_loss:.4f}, Validation AUROC: {val_auroc:.4f}')
    
    # Early stopping
    early_stopping(val_loss, model, epoch)
    if epoch >= 10:
        if early_stopping.early_stop:
            print(f'Early stopping at epoch {epoch+1}')
            break


Epoch 1/100: 100%|██████████| 16590/16590 [04:10<00:00, 66.30batch/s, loss=1.87] 


Epoch [1/100], Loss: 0.4513
Validation Loss: 0.4497, Validation AUROC: 0.5632


Epoch 2/100: 100%|██████████| 16590/16590 [04:08<00:00, 66.63batch/s, loss=0.219]


Epoch [2/100], Loss: 0.4486
Validation Loss: 0.4484, Validation AUROC: 0.5980


Epoch 3/100: 100%|██████████| 16590/16590 [04:10<00:00, 66.20batch/s, loss=0.189]


Epoch [3/100], Loss: 0.4476
Validation Loss: 0.4472, Validation AUROC: 0.6060


Epoch 4/100: 100%|██████████| 16590/16590 [04:08<00:00, 66.69batch/s, loss=0.176]


Epoch [4/100], Loss: 0.4467
Validation Loss: 0.4463, Validation AUROC: 0.6204


Epoch 5/100: 100%|██████████| 16590/16590 [04:10<00:00, 66.32batch/s, loss=1.6]  


Epoch [5/100], Loss: 0.4459
Validation Loss: 0.4456, Validation AUROC: 0.6219


In [30]:

# Store IG scores after training
ig_scores_after_training = model.immunogenicity.ig.clone().detach().cpu()

# Create DataFrame for IG scores
ig_score = {
    'Gene': all_genes,
    'IG Score Before Training': ig_scores_before_training.numpy(),
    'IG Score After Training': ig_scores_after_training.numpy()
}
df = pd.DataFrame(ig_score)

# Calculate the difference and add it as a new column
df['Difference'] = df['IG Score After Training'] - df['IG Score Before Training']

# Sort the DataFrame by the Difference column in descending order
df = df.sort_values(by='Difference', ascending=False)

# Write the sorted DataFrame to a CSV file
output_path = os.path.join(output_dir, 'ig_score_changes.csv')
df.to_csv(output_path, index=False)

# Save the final model
torch.save(model.state_dict(), os.path.join(output_dir, 'final_model.pth'))

print("Training complete. Model and IG scores saved.")

Training complete. Model and IG scores saved.


In [31]:
output_dir

'test/HumanColorectalCancer'